## 환경 준비

아래 셀은 이 노트북에 필요한 Python 패키지가 설치되어 있는지 확인하고,
없으면 자동으로 설치한다. 이미 설치되어 있으면 빠르게 스킵된다.
터미널에서 미리 `uv sync`를 했다면 이 셀은 아무것도 설치하지 않는다.


In [ ]:
# === 의존성 자동 설치 (이미 설치되어 있으면 빠르게 스킵됨) ===
import subprocess, sys

_IMPORT_NAME_OVERRIDES = {
    "scikit-learn": "sklearn",
    "python-dateutil": "dateutil",
    "beautifulsoup4": "bs4",
}


def _ensure_packages(*packages):
    """누락된 패키지만 설치. 이미 있으면 스킵."""
    missing = []
    for pkg in packages:
        name = pkg.split(">=")[0].split("==")[0].split("[")[0]
        import_name = _IMPORT_NAME_OVERRIDES.get(name, name.replace("-", "_"))
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("All packages already installed ✓")

_ensure_packages(
    "pandas", "numpy", "matplotlib", "scikit-learn",
    "pydantic", "python-dateutil", "rich", "tqdm",
    "requests", "beautifulsoup4", "markdownify",
)

# Ollama 서버 연결 확인
import requests
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama connected ✓ models: {models}")
except Exception:
    print("⚠️ Ollama not available — LLM 기능은 규칙 기반으로 폴백됩니다")


# 실제 기술 문서로 에이전트 테스트하기

가상 사내 문서에서는 잘 보이던 패턴이 실제 공개 기술 문서에서는 자주 깨진다. 이유는 문서 길이가 길고, 전문 용어가 많고, 개념이 중첩돼 있으며, 질문이 더 넓은 문맥을 요구하기 때문이다. 이 노트북은 `tech_docs` 프로필을 이용해 demo 데이터와 실제 기술 문서의 retrieval 및 workflow 품질 차이를 직접 비교한다.

## 학습 목표
- 데이터 프로필(data profile) 패턴이 왜 실험 반복성과 재현성에 유리한지 이해한다.
- demo corpus와 tech docs corpus의 retrieval 난이도 차이를 score 분포로 읽을 수 있다.
- 실제 문서에서 어떤 failure type이 더 자주 나타나는지 설명할 수 있다.
- 가상 데이터에서 개발한 시스템을 실제 데이터로 검증하는 과정의 의미를 이해한다.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RuntimeConfig

runtime_config = RuntimeConfig.auto_detect()
print(sys.executable)
{
    'device': runtime_config.device,
    'llm_model': runtime_config.llm_model,
    'llm_available': runtime_config.llm_available,
}

## 구현: 데이터 프로필 시스템

**목적**
        - `load_profile("tech_docs")` 한 줄로 retriever, 문서, eval dataset을 함께 불러오는 구조를 이해한다.

        **핵심 로직**
        - `PROFILES` dict가 데이터셋별 raw_dir, eval_dataset, 설명, 언어를 관리한다.
        - `load_profile()`가 문서 로드, 청킹, retriever 생성, 통계 계산을 한 번에 묶어 반환한다.
        - `tech_docs`는 하위 폴더가 많으므로 `recursive=True`로 문서를 재귀 탐색한다.

        **실제 소스 코드: src/data_profiles.py 전체**
        ```python
        from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
from typing import Any

import pandas as pd

from src.config import RuntimeConfig, get_paths
from src.ingestion import build_demo_index, chunk_document, load_documents
from src.utils import read_json


PROFILES = {
    "demo": {
        "raw_dir": "data/raw/demo",
        "eval_dataset": "data/eval/eval_dataset.json",
        "description": "가상 사내 문서 7개 (영문). 학습/테스트용.",
        "language": "en",
    },
    "tech_docs": {
        "raw_dir": "data/raw/tech_docs",
        "eval_dataset": "data/eval/eval_dataset_tech_docs.json",
        "description": "Anthropic, LangGraph, sentence-transformers 공개 기술 문서.",
        "language": "en",
    },
    "korean_public": {
        "raw_dir": "data/raw/korean_public",
        "eval_dataset": "data/eval/eval_dataset_korean.json",
        "description": "한국어 법령, 정책, 공공데이터.",
        "language": "ko",
    },
}


def _resolve_profile(name: str) -> dict[str, Any]:
    if name not in PROFILES:
        available = ", ".join(sorted(PROFILES))
        raise KeyError(f"Unknown profile '{name}'. Available profiles: {available}")
    return PROFILES[name]


def _profile_paths(name: str) -> tuple[Path, Path]:
    paths = get_paths()
    profile = _resolve_profile(name)
    return paths.root / profile["raw_dir"], paths.root / profile["eval_dataset"]


def _is_recursive_profile(name: str) -> bool:
    return name == "tech_docs"


def _load_profile_materials(name: str) -> dict[str, Any]:
    profile = _resolve_profile(name)
    raw_dir, eval_path = _profile_paths(name)
    recursive = _is_recursive_profile(name)
    documents = load_documents(raw_dir=raw_dir, recursive=recursive)
    chunks = [
        chunk
        for document in documents
        for chunk in chunk_document(document)
    ]
    eval_dataset = read_json(eval_path)
    return {
        "profile": profile,
        "raw_dir": raw_dir,
        "eval_path": eval_path,
        "recursive": recursive,
        "documents": documents,
        "chunks": chunks,
        "eval_dataset": eval_dataset,
    }


def list_profiles() -> pd.DataFrame:
    """사용 가능한 프로필 목록."""
    rows = [{"name": name, **profile} for name, profile in PROFILES.items()]
    return pd.DataFrame(
        rows,
        columns=["name", "raw_dir", "eval_dataset", "description", "language"],
    )


def load_profile(name: str, backend: str = "tfidf", persist: bool = False) -> dict[str, Any]:
    """Load a profile with retriever, corpus, evaluation set, and summary stats."""
    materials = _load_profile_materials(name)
    runtime_config = RuntimeConfig.auto_detect()
    retriever = build_demo_index(
        raw_dir=materials["raw_dir"],
        persist=persist,
        backend=backend,
        recursive=materials["recursive"],
    )

    stats = {
        "name": name,
        "language": materials["profile"]["language"],
        "backend": backend,
        "recursive": materials["recursive"],
        "document_count": len(materials["documents"]),
        "chunk_count": len(materials["chunks"]),
        "eval_question_count": len(materials["eval_dataset"]),
        "avg_document_chars": round(
            sum(len(document["text"]) for document in materials["documents"]) / len(materials["documents"]),
            1,
        )
        if materials["documents"]
        else 0.0,
        "avg_chunk_chars": round(
            sum(len(chunk["text"]) for chunk in materials["chunks"]) / len(materials["chunks"]),
            1,
        )
        if materials["chunks"]
        else 0.0,
    }

    return {
        "name": name,
        "retriever": retriever,
        "eval_dataset": materials["eval_dataset"],
        "documents": materials["documents"],
        "chunks": materials["chunks"],
        "config": {
            **materials["profile"],
            "backend": backend,
            "persist": persist,
            "recursive": materials["recursive"],
            "runtime": asdict(runtime_config),
        },
        "stats": stats,
    }


def compare_profiles(*names: str) -> pd.DataFrame:
    """여러 프로필의 데이터 규모와 평가셋 크기를 비교."""
    selected_names = names or tuple(PROFILES.keys())
    rows: list[dict[str, Any]] = []
    for name in selected_names:
        loaded = load_profile(name, persist=False)
        row = {"name": loaded["name"], **loaded["stats"]}
        rows.append(row)
    return pd.DataFrame(rows)
        ```

        **실제 소스 코드: load_documents(recursive) — src/ingestion.py**
        ```python
        def load_documents(raw_dir: Path | None = None, recursive: bool = False) -> list[dict[str, str]]:
    paths = get_paths()
    source_dir = raw_dir or paths.raw_dir
    if raw_dir is None and (source_dir / "demo").exists():
        source_dir = source_dir / "demo"
    documents: list[dict[str, str]] = []

    iterator = source_dir.rglob("*") if recursive else source_dir.iterdir()
    for path in sorted(iterator):
        if not path.is_file():
            continue
        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        if path.parent == source_dir:
            source = path.name
            doc_id = path.stem
        else:
            relative_path = path.relative_to(source_dir)
            source = str(relative_path)
            doc_id = "__".join(relative_path.with_suffix("").parts)
        documents.append(
            {
                "doc_id": doc_id,
                "source": source,
                "text": path.read_text(),
            }
        )

    return documents
        ```

        **코드 읽기 포인트**
        - `_is_recursive_profile(name)`가 `tech_docs`에만 재귀 로딩을 켜는 이유는, 도메인별 하위 폴더 구조를 그대로 보존하기 위해서다.
        - `load_profile()`는 retriever뿐 아니라 `documents`, `chunks`, `eval_dataset`, `stats`를 같이 반환하므로 notebook이 반복되는 준비 코드를 줄일 수 있다.
        - `load_documents()`의 `source` 생성 규칙 덕분에 `anthropic/...`, `langgraph/...`처럼 하위 경로가 citation과 eval source matching에 그대로 쓰인다.

        **결과 해석 가이드**
        - `document_count`와 `chunk_count`가 demo보다 크면 retrieval 검색 공간이 넓어진다는 뜻이다.
        - `avg_document_chars`, `avg_chunk_chars`가 커질수록 긴 문서와 전문 용어가 retrieval/synthesis 난이도를 높일 가능성이 있다.

        **💡 면접 포인트**
        - 프로필 패턴을 쓰면 같은 workflow를 여러 corpus에 재사용하면서도 실험 준비 코드를 단순하게 유지할 수 있다.
        - 실제 데이터로 갈수록 recursive loading, source naming, eval dataset 연결 같은 데이터 관리 계층이 중요해진다.

### src 코드 펼침: `load_profile()`

```python
def load_profile(name: str, backend: str = "tfidf", persist: bool = False) -> dict[str, Any]:
    materials = _load_profile_materials(name)
    runtime_config = RuntimeConfig.auto_detect()
    retriever = build_demo_index(
        raw_dir=materials["raw_dir"],
        persist=persist,
        backend=backend,
        recursive=materials["recursive"],
    )

    stats = {
        "name": name,
        "language": materials["profile"]["language"],
        "backend": backend,
        "recursive": materials["recursive"],
        "document_count": len(materials["documents"]),
        "chunk_count": len(materials["chunks"]),
        "eval_question_count": len(materials["eval_dataset"]),
        ...
    }

    return {
        "name": name,
        "retriever": retriever,
        "eval_dataset": materials["eval_dataset"],
        "documents": materials["documents"],
        "chunks": materials["chunks"],
        "config": {...},
        "stats": stats,
    }
```

- `load_profile("tech_docs")` 한 줄이 가능한 이유는 profile 메타데이터, 문서 로딩, 청킹, retriever 생성, eval dataset 연결을 이 함수가 한 번에 묶기 때문이다.
- `backend`, `persist`, `recursive` 같은 실행 옵션은 profile 이름과 함께 config에 다시 담아 반환한다. 그래서 notebook에서는 지금 어떤 데이터와 어떤 인덱스를 보고 있는지 추적하기 쉽다.
- 결국 profile 패턴은 "데이터셋을 바꾸더라도 notebook 코드는 최대한 그대로 유지"하려는 장치다.

### src 코드 펼침: `load_documents(recursive=True)`

```python
def load_documents(raw_dir: Path | None = None, recursive: bool = False) -> list[dict[str, str]]:
    source_dir = raw_dir or paths.raw_dir
    ...
    iterator = source_dir.rglob("*") if recursive else source_dir.iterdir()
    for path in sorted(iterator):
        if not path.is_file():
            continue
        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        if path.parent == source_dir:
            source = path.name
            doc_id = path.stem
        else:
            relative_path = path.relative_to(source_dir)
            source = str(relative_path)
            doc_id = "__".join(relative_path.with_suffix("").parts)
        documents.append({"doc_id": doc_id, "source": source, "text": path.read_text()})
```

- `recursive=False`이면 현재 디렉토리만 훑고, `recursive=True`이면 `rglob("*")`로 하위 폴더까지 재귀 탐색한다. `tech_docs`처럼 `anthropic/`, `langgraph/`, `sentence_transformers/`로 나뉜 코퍼스에서 필수다.
- 하위 폴더 문서는 `source`에 상대 경로가 들어간다. 예를 들어 `anthropic/tool-use.md`처럼 저장되어 나중에 citation이나 에러 분석에서 도메인을 바로 읽을 수 있다.
- `doc_id`를 `__`로 이어 붙이는 이유는 파일 시스템 경로 구분자를 데이터 키에 그대로 넣지 않기 위해서다. retriever와 trace가 모두 안정적으로 같은 식별자를 쓸 수 있다.


In [ ]:
import pandas as pd

from src.data_profiles import compare_profiles, load_profile

tech_profile = load_profile('tech_docs', persist=False)
comparison_stats = compare_profiles('demo', 'tech_docs')

print('Tech docs profile stats')
print(tech_profile['stats'])
comparison_stats[['name', 'language', 'document_count', 'chunk_count', 'eval_question_count', 'avg_document_chars', 'avg_chunk_chars']]

## Demo vs Tech Docs 검색 품질 비교

**목적**
- 가상 문서와 실제 기술 문서가 retrieval 관점에서 얼마나 다른지 score 분포로 비교한다.

**핵심 로직**
- 같은 query type 하나를 골라 demo와 tech_docs에서 각각 search를 수행한다.
- rank별 score를 선으로 그려, 상위 몇 개 결과가 얼마나 빠르게 점수가 떨어지는지 본다.

**주요 파라미터**
- `query_type_to_compare`: 이번에는 `summary`를 선택해 긴 문서 검색 난이도를 본다.
- `top_k=5`: 상위 5개 근거 후보를 비교한다.

기술 문서는 어려운 이유가 명확하다. 길이가 길고, 같은 개념을 여러 문서가 다른 표현으로 설명하며, 전문 용어가 많고, 질문과 직접적인 표면 단어가 덜 겹칠 수 있다. 그래서 demo에서 잘 되던 lexical retrieval이 실제 문서에서는 더 흔들릴 수 있다.

**결과 해석 가이드**
- rank 1과 rank 2 점수 차가 크면 retrieval confidence가 높은 편이다.
- score가 완만하게 떨어지면 관련 문서가 여러 개 섞여 있다는 뜻이고, 요약/다중근거 질문에서는 오히려 좋은 신호일 수 있다.
- tech_docs 쪽 상위 score가 낮게 나오더라도 꼭 실패는 아니다. 실제 문서가 더 어려워 분포 자체가 눌릴 수 있기 때문이다.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.data_profiles import load_profile


demo_profile = load_profile('demo', persist=False)
demo_dataset = pd.DataFrame(demo_profile['eval_dataset']).sort_values(['question_type', 'id'])
tech_dataset = pd.DataFrame(tech_profile['eval_dataset']).sort_values(['question_type', 'id'])

query_type_to_compare = 'summary'
demo_query = demo_dataset[demo_dataset['question_type'] == query_type_to_compare].iloc[0]['question']
tech_query = tech_dataset[tech_dataset['question_type'] == query_type_to_compare].iloc[0]['question']

demo_results = demo_profile['retriever'].search(demo_query, top_k=5)
tech_results = tech_profile['retriever'].search(tech_query, top_k=5)

score_frame = pd.DataFrame(
    [
        {'profile': 'demo', 'rank': index + 1, 'score': item['score'], 'source': item['source'], 'question': demo_query}
        for index, item in enumerate(demo_results)
    ]
    + [
        {'profile': 'tech_docs', 'rank': index + 1, 'score': item['score'], 'source': item['source'], 'question': tech_query}
        for index, item in enumerate(tech_results)
    ]
)

fig, ax = plt.subplots(figsize=(8, 4))
for profile_name, frame in score_frame.groupby('profile'):
    ax.plot(frame['rank'], frame['score'], marker='o', label=profile_name)
ax.set_title(f'Retrieval score distribution for query_type={query_type_to_compare}')
ax.set_xlabel('rank')
ax.set_ylabel('score')
ax.legend()
plt.tight_layout()
plt.show()

score_frame[['profile', 'rank', 'score', 'source']]

## 결과 해석

여기서는 숫자 절대값보다 두 프로필의 score shape를 비교하는 것이 중요하다. demo는 짧고 선명한 문서가 많아 상위 점수가 높고 분리가 잘 되는 경우가 많다. 반면 tech_docs는 관련 문서가 여러 개 경쟁하거나, 같은 개념이 길게 설명돼 점수가 분산될 수 있다.

따라서 실제 문서 실험에서는 retrieval score가 조금 낮아졌다는 이유만으로 retriever를 실패라고 단정하지 말고, 상위 source가 질문 의도와 맞는지 먼저 읽어야 한다.


## 실험: 워크플로우 실행 (`use_llm=True`)

**목적**
- Anthropic, LangGraph, sentence-transformers 도메인 질문을 실제로 workflow에 태워 보고, trace가 어떤 모양으로 남는지 확인한다.

**핵심 로직**
- 각 도메인 prefix에 대응하는 평가 질문 하나씩 골라 `run_workflow(use_llm=True)`를 실행한다.
- Ollama가 살아 있으면 live path, 아니면 fallback path로 동작한다.
- 대표 trace를 열어 실제 데이터에서 어떤 node가 부담을 많이 지는지 본다.

**주요 파라미터**
- `selected_questions`: 도메인별 대표 질문 샘플이다.
- `llm_path`: live / fallback / server_unavailable 구분값이다.
- `trace_steps`: 실제 실행된 workflow 단계 수다.

**결과 해석 가이드**
- `coverage_score`가 낮으면 기술 문서의 길이와 용어 밀도 때문에 synthesis/verifier가 어려움을 겪었을 가능성이 있다.
- `trace_steps`는 대개 비슷해야 하지만, 도구 호출이나 예외 경로에 따라 달라질 수 있다.
- trace preview에서 retrieval output source가 질문 도메인과 맞는지 먼저 확인하는 것이 좋다.

**💡 면접 포인트**
- 가상 데이터에서 잘 되던 workflow도 실제 기술 문서에서는 retrieval miss, chunking 문제, synthesis 품질 저하가 더 쉽게 드러난다.
- 실제 데이터 검증은 모델 시연이 아니라 failure discovery 단계로 봐야 한다.

### src 코드 펼침: `display_trace()`

```python
def display_trace(trace: list[dict[str, Any]], render: bool = True) -> pd.DataFrame:
    frame = trace_to_debug_frame(trace)
    if render:
        try:
            from IPython.display import display
            display(frame)
        except Exception:
            print(frame.to_string(index=False))
    return frame
```

```python
def trace_to_debug_frame(trace: list[dict[str, Any]]) -> pd.DataFrame:
    previous_timestamp: datetime | None = None
    rows: list[dict[str, Any]] = []
    for index, entry in enumerate(trace, start=1):
        current_timestamp = _parse_timestamp(str(entry.get("timestamp", "")))
        latency = entry.get("latency")
        if not isinstance(latency, (int, float)) and previous_timestamp is not None and current_timestamp is not None:
            latency = round((current_timestamp - previous_timestamp).total_seconds(), 6)
        rows.append({
            "step": index,
            "node": str(entry.get("node", "")),
            "latency": round(float(latency), 6) if isinstance(latency, (int, float)) else None,
            "inputs": _serialize(entry.get("inputs", {})),
            "outputs": _serialize(entry.get("outputs", entry.get("payload", {}))),
            "timestamp": str(entry.get("timestamp", "")),
        })
    return pd.DataFrame(rows, columns=["step", "node", "latency", "inputs", "outputs", "timestamp"])
```

- trace는 원래 JSON list라 사람이 읽기 어렵다. `trace_to_debug_frame()`이 이를 step별 행(row)으로 바꿔 notebook에서 읽기 좋은 표로 만든다.
- `latency`가 trace entry에 직접 있으면 그 값을 쓰고, 없으면 앞뒤 `timestamp` 차이로 계산한다. 즉, 구 trace와 신 trace를 모두 읽을 수 있게 만든 호환성 레이어다.
- `inputs`, `outputs`는 `_serialize()`로 길이를 잘라 넣는다. 디버깅에서 중요한 건 전체 JSON 원문보다 "이 노드에 무엇이 들어갔고 무엇이 나왔는가"를 빠르게 보는 것이다.


In [ ]:
import warnings

import pandas as pd
from IPython.display import display

from src.llm_client import OllamaClient
from src.trace_debug import display_trace
from src.workflow import run_workflow

tech_client = OllamaClient()
tech_llm_live = tech_client.is_available()
tech_dataset_frame = pd.DataFrame(tech_profile['eval_dataset'])

selected_questions = []
for domain_prefix in ['anthropic/', 'langgraph/', 'sentence_transformers/']:
    for row in tech_dataset_frame.to_dict(orient='records'):
        expected_sources = row.get('expected_sources', [])
        if expected_sources and expected_sources[0].startswith(domain_prefix):
            selected_questions.append((domain_prefix.rstrip('/'), row))
            break

states = {}
rows = []
for domain_name, sample in selected_questions:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        state = run_workflow(
            sample['question'],
            retriever=tech_profile['retriever'],
            use_llm=True,
            llm_client=tech_client,
        )
    states[domain_name] = state
    rows.append(
        {
            'domain': domain_name,
            'question_id': sample['id'],
            'question_type': sample['question_type'],
            'final_status': state['final_status'],
            'coverage_score': state['verification_result'].coverage_score,
            'trace_steps': len(state['trace']),
            'llm_path': 'live' if tech_llm_live and not caught else ('fallback' if caught else 'server_unavailable'),
            'answer_preview': ' '.join(state['final_answer'].split())[:160],
        }
    )

domain_run_summary = pd.DataFrame(rows)
display(domain_run_summary)
print('Anthropic trace preview')
display(display_trace(states['anthropic']['trace'], render=False))

## 결과 해석: Demo vs Tech Docs 평가

**목적**
- demo와 tech_docs에서 같은 workflow를 돌렸을 때 품질 지표가 어떻게 달라지는지 평균 수준에서 본다.

**핵심 로직**
- 각 프로필에서 query type별 균형 샘플을 뽑아 `answer_correctness`, `retrieval_hit_rate`, `grounding_pass_rate`, `latency_seconds`를 계산한다.

**결과 해석 가이드**
- `retrieval_hit_rate`가 유지되는데 `answer_correctness`만 떨어지면 실제 문서에서 synthesis가 더 어려워졌을 가능성이 크다.
- `grounding_pass_rate`가 떨어지면 verifier가 실제 문서의 더 긴 답변과 복잡한 근거를 엄격하게 보고 있다는 뜻일 수 있다.
- `latency_seconds`가 오르면 문서 길이 증가와 LLM 답변 길이 증가가 같이 영향을 줬을 수 있다.


In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from src.evaluator import retrieval_hit_rate, score_answer
from src.llm_client import OllamaClient
from src.workflow import run_workflow

comparison_client = OllamaClient()
comparison_llm_live = comparison_client.is_available()

def sample_balanced_questions(dataset: list[dict], count_per_type: int = 2) -> list[dict]:
    frame = pd.DataFrame(dataset).sort_values(['question_type', 'id'])
    return (
        frame.groupby('question_type', as_index=False, group_keys=False)
        .head(count_per_type)
        .to_dict(orient='records')
    )


def evaluate_profile_subset(profile_name: str, profile: dict, sample_rows: list[dict]) -> pd.DataFrame:
    rows = []
    for sample in sample_rows:
        start = time.perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            state = run_workflow(
                sample['question'],
                retriever=profile['retriever'],
                use_llm=True,
                llm_client=comparison_client,
            )
        latency = round(time.perf_counter() - start, 4)
        rows.append(
            {
                'profile': profile_name,
                'question_id': sample['id'],
                'question_type': sample['question_type'],
                'answer_correctness': score_answer(
                    state['final_answer'],
                    sample['gold_answer'],
                    state['final_status'],
                    sample.get('expected_status', 'answered'),
                ),
                'retrieval_hit_rate': retrieval_hit_rate(
                    state['retrieved_docs'],
                    sample.get('expected_sources', []),
                ),
                'grounding_pass_rate': float(state['verification_result'].is_grounded),
                'latency_seconds': latency,
                'path_mode': 'live' if comparison_llm_live and not caught else ('fallback' if caught else 'server_unavailable'),
            }
        )
    return pd.DataFrame(rows)


demo_subset = sample_balanced_questions(demo_profile['eval_dataset'])
tech_subset = sample_balanced_questions(tech_profile['eval_dataset'])

demo_eval = evaluate_profile_subset('demo', demo_profile, demo_subset)
tech_eval = evaluate_profile_subset('tech_docs', tech_profile, tech_subset)
profile_eval = pd.concat([demo_eval, tech_eval], ignore_index=True)
profile_summary = (
    profile_eval.groupby('profile', as_index=False)
    .agg(
        answer_correctness=('answer_correctness', 'mean'),
        retrieval_hit_rate=('retrieval_hit_rate', 'mean'),
        grounding_pass_rate=('grounding_pass_rate', 'mean'),
        latency_seconds=('latency_seconds', 'mean'),
    )
    .round(3)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
profile_summary.plot(x='profile', y=['answer_correctness', 'grounding_pass_rate'], kind='bar', ax=axes[0], color=['#4C78A8', '#54A24B'])
axes[0].set_ylim(0, 1)
axes[0].set_title('Quality Metrics')
axes[0].set_ylabel('score')
profile_summary.plot(x='profile', y='latency_seconds', kind='bar', ax=axes[1], color='#F58518', legend=False)
axes[1].set_title('Average Latency')
axes[1].set_ylabel('seconds')
plt.tight_layout()
plt.show()

profile_summary

## Failure 분석

**목적**
- 실제 기술 문서에서 demo에 없던 failure type이 추가로 나타나는지 본다.

**핵심 로직**
- profile별 evaluation 결과를 failure_analyzer에 넣어 failure type 분포를 만든다.
- demo에는 없고 tech_docs에만 있는 failure type을 추린다.

**주요 파라미터**
- `tech_only_failures`: 실제 기술 문서에서만 드러난 failure type 목록이다.
- `failure_distribution`: profile과 failure type별 발생 횟수다.

**결과 해석 가이드**
- `tech_only_failures`가 있다면 가상 데이터 기반 개발 단계에서는 보이지 않던 운영 리스크가 드러난 것이다.
- retrieval_noise, missing_decomposition 같은 failure가 늘면 실제 데이터에서는 더 정교한 chunking과 planning이 필요하다는 뜻이다.

**💡 면접 포인트**
- 가상 데이터에서 개발하고 실제 데이터에서 failure taxonomy로 검증하면, 개선 우선순위를 훨씬 설득력 있게 제시할 수 있다.
- 실제 기술 문서는 retrieval 자체보다도 long-context synthesis와 source attribution에서 더 많은 문제를 드러내는 경우가 많다.

### src 코드 펼침: `analyze_failures()`

```python
def analyze_failures(results_df: pd.DataFrame) -> dict[str, Any]:
    if results_df.empty:
        return {
            "total_failure_instances": 0,
            "failure_distribution": {},
            "stage_distribution": {},
            "severity_distribution": {},
            "top_improvement_actions": [],
        }

    failure_records = _records_with_failures(results_df)
    failure_distribution = Counter(failure for failure, _ in failure_records)
    stage_distribution = Counter(metadata["stage"] for _, metadata in failure_records)
    severity_distribution = Counter(metadata["severity"] for _, metadata in failure_records)
    mitigation_distribution = Counter(failure_mitigation(failure) for failure, _ in failure_records)

    top_improvement_actions = [
        {"mitigation": mitigation, "count": count}
        for mitigation, count in mitigation_distribution.most_common(5)
    ]

    return {
        "total_failure_instances": sum(failure_distribution.values()),
        "failure_distribution": dict(sorted(failure_distribution.items())),
        "stage_distribution": dict(sorted(stage_distribution.items())),
        "severity_distribution": dict(sorted(severity_distribution.items())),
        "top_improvement_actions": top_improvement_actions,
    }
```

- 이 함수는 개별 실패를 보는 데서 멈추지 않고, 실패를 조직적으로 집계한다. 실무에서는 한 건의 예외보다 "어떤 단계에서 반복적으로 많이 깨지는가"가 더 중요하다.
- `failure_distribution`은 failure type 빈도, `stage_distribution`은 어느 노드 단계에서 문제가 집중되는지, `severity_distribution`은 문제의 심각도를 보여준다.
- `mitigation_distribution`은 단순 통계 이상으로 실무 가치가 크다. 실패 유형을 "어떤 개선 작업을 우선할 것인가"로 바로 연결하기 때문이다.
- 그래서 마지막 `top_improvement_actions`는 단순 보고서가 아니라 backlog 우선순위의 초안이라고 볼 수 있다.


In [ ]:
import pandas as pd

from src.failure_analyzer import analyze_failures, classify_failure

failure_rows = []
for record in profile_eval.to_dict(orient='records'):
    enriched = dict(record)
    enriched.update(
        {
            'expected_status': 'answered',
            'predicted_status': 'answered' if record['answer_correctness'] > 0 else 'abstained',
            'grounding_pass': bool(record['grounding_pass_rate']),
            'predicted_question_type': record['question_type'],
            'expected_question_type': record['question_type'],
            'average_steps': 9.0,
            'citations': [],
            'expected_sources': [],
            'trace': [],
            'errors': [],
            'system': 'agent_workflow',
        }
    )
    failures = classify_failure(enriched)
    failure_rows.append({
        'profile': record['profile'],
        'question_id': record['question_id'],
        'failure_types': failures or ['none'],
    })

failure_frame = pd.DataFrame(failure_rows).explode('failure_types').rename(columns={'failure_types': 'failure_type'})
summary_input = profile_eval.rename(columns={'profile': 'system'})
demo_failure_types = set(failure_frame[failure_frame['profile'] == 'demo']['failure_type']) - {'none'}
tech_failure_types = set(failure_frame[failure_frame['profile'] == 'tech_docs']['failure_type']) - {'none'}
tech_only_failures = sorted(tech_failure_types - demo_failure_types)

failure_distribution = (
    failure_frame.groupby(['profile', 'failure_type'], as_index=False)
    .size()
    .sort_values(['profile', 'size'], ascending=[True, False])
)

analysis_snapshot = {
    'demo': analyze_failures(demo_eval.rename(columns={'profile': 'system'})),
    'tech_docs': analyze_failures(tech_eval.rename(columns={'profile': 'system'})),
}

print('Tech-only failure types:', tech_only_failures)
failure_distribution

## 핵심 정리

이 노트북을 통해 실제 기술 문서는 demo 데이터보다 훨씬 까다로운 실험장이라는 점을 확인했다. 문서 수가 많고, 길고, 전문 용어가 많아 retrieval score가 더 분산되고, workflow 전체에서 synthesis와 grounding verification 부담도 커진다. `data_profiles` 패턴은 이런 서로 다른 corpus를 같은 인터페이스로 비교하게 해 주는 핵심 장치다.

**💡 면접 포인트**
- 가상 데이터로 빠르게 개발한 뒤, 실제 기술 문서로 검증하면서 retrieval miss·chunking·source attribution 이슈를 발견했다고 설명할 수 있다.
- 데이터 프로필 패턴 덕분에 corpus만 바꿔도 같은 workflow와 evaluator를 재사용할 수 있어 실험 생산성이 높아진다.
- 실제 데이터 검증은 단순 성능 측정이 아니라, 어떤 failure type이 새롭게 나타나는지 찾는 과정이다.
